# 파노라마 합성 및 객체 검출

> **실험 목표**: SIFT, ORB, ALIKED의 객체 검출 정확도와 실행 시간을 비교하고, 기준 물체의 회전·크기·조명 변화에 대한 강건성을 평가합니다.

> **검출기 변경**: 과제 안내의 AKAZE는 현재 사용 중인 OpenCV 5.0 환경에서 사용할 수 없어 ALIKED로 대체했습니다. ALIKED는 딥러닝 기반 특징점과 LightGlue 매칭을 사용하며, 본 보고서에서는 SIFT·ORB와 함께 성능을 비교합니다.

## 1. 실험 개요

연속 사진을 합성한 파노라마에서 기준 물체의 위치를 찾습니다. 각 방법은 특징점 추출·매칭, Lowe Ratio Test 또는 LightGlue 매칭, RANSAC 기반 호모그래피를 거쳐 검출 결과를 만듭니다.

### 매칭 및 기하 검증 방법

- **Lowe Ratio Test**: SIFT·ORB의 최근접·차근접 매칭 거리를 비교해 모호한 매칭을 제거합니다.
- **LightGlue 매칭**: ALIKED 특징점 사이의 대응 관계를 신경망 기반으로 판단해 신뢰도 높은 매칭을 선택합니다.
- **RANSAC 기반 호모그래피**: 잘못된 매칭(Outlier)을 제외하고, 남은 Inlier로 기준 물체의 평면 좌표 변환과 Polygon 위치를 계산합니다.

## 2. 데이터 구성 및 확장 방식

- 기준 물체: `data/objects/set01/target.jpg`, `set02/target.jpg`, …
- 완성 파노라마: `results/panorama/SIFT/panorama1.jpg`, `ORB/panorama1.jpg`, `ALIKED/panorama1.jpg`
- 파일 번호가 같은 기준 물체와 방법별 파노라마를 자동 연결합니다.
- 각 방법은 자신이 합성한 파노라마에서 객체 검출까지 수행하므로 전체 파이프라인을 비교합니다.
- 현재는 1세트여도, 이후 5세트를 추가하면 코드 수정 없이 전체 케이스에 같은 실험을 적용합니다.

## 3. 평가 기준

- **검출 성공률**: 전체 케이스 중 객체 위치 검출에 성공한 비율
- **평균 탐색 시간**: 특징점 추출과 Coarse-to-Fine 타일 탐색에 걸린 시간
- **매칭 신뢰도**: RANSAC Inlier 수와 Inlier 비율
- **변형 조건**: 기준 상태, 회전 ±30°, 크기 75%·125%, 밝기 70%·130%


### 모듈별 역할

- `src/stitcher.py`: 연속 사진의 특징점 매칭, RANSAC, 호모그래피와 블렌딩으로 파노라마를 합성합니다.
- `src/object_finder.py`: 기준 물체 특징점을 재사용해 Coarse-to-Fine 타일 탐색을 수행하고, 최종 호모그래피를 파노라마 좌표의 Polygon으로 표시합니다.
- `src/pipeline.py`: 합성·검출 모듈을 순서대로 실행하고 이미지·요약 JSON을 저장하는 전체 파이프라인 진입점입니다.
- `src/main.py`: 파노라마 합성 또는 객체 검출 기능을 개별로 확인하는 테스트·디버그 도구입니다.
- `src/project_paths.py`: 데이터와 결과물의 공통 경로, 객체 검출 결과 폴더 규칙을 관리합니다.


In [ ]:
# ==========================================
# 라이브러리 및 프로젝트 모듈 Import
# - 노트북에서 사용하는 공통 설정과 실행 함수를 한곳에서 준비
# ==========================================
import importlib
import logging
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

import src.experiment_runner as experiment_runner
import src.result_cleanup as result_cleanup
import src.settings as settings
import src.stitcher as stitcher

importlib.reload(settings)
importlib.reload(experiment_runner)
importlib.reload(result_cleanup)
importlib.reload(stitcher)

from src.experiment_runner import (
    collect_panorama_summary,
    create_detection_matching_summary_figure,
    create_detection_qualitative_figure,
    create_detection_summary_chart_figure,
    create_detection_summary_table_figure,
    create_panorama_summary_figure,
    discover_test_cases,
    get_experiment_settings,
    read_image_for_display,
    run_batch,
    save_experiment_csv,
    summarize_records,
)
from src.logger import logger
from src.project_paths import (
    OBJECT_DIR,
    PANORAMA_RESULT_DIR,
    project_path,
)
from src.result_cleanup import clear_results
from src.settings import PATH_CONFIG
from src.stitcher import stitch_panorama

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 노트북 로그 설정
# - 화면과 results/logs/opencv.log에는 ERROR 이상만 기록
# ==========================================
NOTEBOOK_LOG_LEVEL = logging.ERROR
logger.setLevel(NOTEBOOK_LOG_LEVEL)
logger.propagate = False
for handler in logger.handlers:
    handler.setLevel(NOTEBOOK_LOG_LEVEL)

cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)

EXPERIMENT = get_experiment_settings()

In [ ]:
# ==========================================
# 이전 결과 정리
# - opencv.log와 .gitkeep은 보존
# - False면 삭제 대상만 미리 확인
# ==========================================
DELETE_PREVIOUS_RESULTS = True

cleanup_targets = clear_results(dry_run=not DELETE_PREVIOUS_RESULTS)
print('정리 대상 수:', len(cleanup_targets))
if DELETE_PREVIOUS_RESULTS:
    print('이전 결과 삭제 완료')
else:
    print('미리보기 모드입니다. 삭제하려면 True로 변경하세요.')

## 4. 파노라마 합성 실험

`data/set01`, `data/set02` 형식의 연속 사진 세트를 자동 탐색한다. 각 세트는
SIFT, ORB, ALIKED 방법으로 합성되며 결과는 `results/panorama/세트명/방법/`에 저장된다.

In [ ]:
# ==========================================
# stitcher.py 기반 파노라마 합성 실행
# - data/set01, data/set02 ... 세트 폴더 자동 탐색
# - 세트 번호를 객체 검출 기준 물체 번호와 연결
# ==========================================
source_root = project_path(PATH_CONFIG['panorama_source_dir'])
source_sets = sorted(
    (path for path in source_root.glob('set*') if path.is_dir()),
    key=lambda path: path.name,
)

if not source_sets:
    raise FileNotFoundError(
        'data 아래에서 set01, set02 형식의 파노라마 세트 폴더를 찾을 수 없습니다.'
    )

pipeline_inputs = []
for source_dir in source_sets:
    match = re.search(r'\d+$', source_dir.name)
    if match is None:
        raise ValueError(
            f'파노라마 세트 폴더명에 번호가 필요합니다: {source_dir.name}'
        )
    pipeline_inputs.append((str(int(match.group())), source_dir))

panorama_results = []
for case_id, source_dir in pipeline_inputs:
    for method in EXPERIMENT.methods:
        print(f'실행 중: set={source_dir.name}, method={method}')
        result = stitch_panorama(
            input_dir=source_dir,
            method=method,
            max_side=EXPERIMENT.panorama_max_side,
            set_name=f'set{int(case_id):02d}',
        )
        panorama_results.append(result)

print(f'파노라마 합성 실행 완료: {len(panorama_results)}회')

### 4.1 방법별 파노라마 합성 매칭 통계

인접 사진 쌍의 원시 매칭 수, 필터링 후 매칭 수, RANSAC Inlier 수와 Inlier 비율을 비교한다.

In [ ]:
# ==========================================
# 파노라마 합성 통계 수집 및 비교
# - 세트·방법별 matching_stats.csv 자동 탐색
# ==========================================
panorama_summary_rows = collect_panorama_summary(
    panorama_result_dir=PANORAMA_RESULT_DIR,
    methods=EXPERIMENT.methods,
)
create_panorama_summary_figure(panorama_summary_rows, plt)
plt.show()

In [ ]:
# ==========================================
# 방법별 인접 사진 필터링 매칭 결과
# - 첫 번째 사진 쌍(01.jpg <-> 02.jpg)의 필터링 결과를 비교
# ==========================================
visual_set = sorted({row['set'] for row in panorama_summary_rows})[0]
visual_rows = [
    row for row in panorama_summary_rows
    if row['set'] == visual_set
]

fig, axes = plt.subplots(
    1, len(visual_rows),
    figsize=(7 * len(visual_rows), 4),
    constrained_layout=True,
)
axes = np.atleast_1d(axes)

for axis, row in zip(axes, visual_rows):
    matching_path = (
        PANORAMA_RESULT_DIR / row['set'] / row['method']
        / 'matching' / 'pair_01_02_filtered.png'
    )
    if matching_path.is_file():
        axis.imshow(read_image_for_display(matching_path, max_width=900))
    else:
        axis.text(
            0.5, 0.5, '필터링 매칭 이미지가 없습니다.',
            ha='center', va='center', wrap=True,
        )
    axis.set_title(f"{row['method']} / pair_01_02 필터링 매칭")
    axis.set_xticks([])
    axis.set_yticks([])

fig.suptitle(f'방법별 인접 사진 필터링 매칭: {visual_set}', fontsize=16)
plt.show()

In [ ]:
# ==========================================
# 파노라마 매칭 이미지 전체 확인
# - raw / filtered / ransac 단계별 이미지를 모두 표시
# ==========================================
MATCHING_VISUAL_SET = 'set01'
MATCHING_VISUAL_METHOD = 'ALIKED'

matching_dir = (
    PANORAMA_RESULT_DIR / MATCHING_VISUAL_SET
    / MATCHING_VISUAL_METHOD / 'matching'
)
matching_paths = sorted(matching_dir.glob('*.png'))
if not matching_paths:
    raise FileNotFoundError(
        f'매칭 이미지를 찾을 수 없습니다: {matching_dir}'
    )

column_count = 3
row_count = (len(matching_paths) + column_count - 1) // column_count
fig, axes = plt.subplots(
    row_count, column_count,
    figsize=(18, 5 * row_count),
    constrained_layout=True,
)
axes = np.atleast_1d(axes).ravel()

for axis, matching_path in zip(axes, matching_paths):
    axis.imshow(read_image_for_display(matching_path, max_width=800))
    axis.set_title(matching_path.stem.replace('_', ' '), fontsize=11)
    axis.set_xticks([])
    axis.set_yticks([])

for axis in axes[len(matching_paths):]:
    axis.axis('off')

fig.suptitle(
    f'{MATCHING_VISUAL_SET} / {MATCHING_VISUAL_METHOD} 파노라마 매칭 단계별 결과',
    fontsize=16,
)
plt.show()

In [ ]:
# ==========================================
# 방법별 파노라마 합성 결과 시각 비교
# - 파노라마와 합성 윤곽선 이미지를 방법별로 표시
# ==========================================
visual_set = sorted({row['set'] for row in panorama_summary_rows})[0]
visual_rows = [
    row for row in panorama_summary_rows
    if row['set'] == visual_set
]

fig, axes = plt.subplots(
    2, len(visual_rows),
    figsize=(7 * len(visual_rows), 9),
    constrained_layout=True,
)
axes = np.asarray(axes, dtype=object).reshape(2, len(visual_rows))

for column, row in enumerate(visual_rows):
    panorama_path = Path(row['panorama_path'])
    outline_path = panorama_path.parent / 'panorama_outline.jpg'

    axes[0, column].imshow(
        read_image_for_display(panorama_path, max_width=900)
    )
    axes[0, column].set_title(f"{row['method']} 파노라마")

    if outline_path.is_file():
        axes[1, column].imshow(
            read_image_for_display(outline_path, max_width=900)
        )
    else:
        axes[1, column].text(
            0.5, 0.5, '합성 윤곽선 이미지가 없습니다.',
            ha='center', va='center', wrap=True,
        )
    axes[1, column].set_title(f"{row['method']} 합성 윤곽선")

    for axis in axes[:, column]:
        axis.set_xticks([])
        axis.set_yticks([])

fig.suptitle(f'방법별 파노라마 합성 결과: {visual_set}', fontsize=18)
plt.show()

## 5. 객체 검출 탐색 설정

타일 크기, 탐색 간격, 후보 수, 조기 종료 기준은 `config.json`의 객체 검출 설정을 사용한다.
기준 물체에는 기본 조건과 회전·크기·조명 변화 6개 조건을 적용한다.

In [ ]:
# ==========================================
# 객체 검출 공통 환경 확인
# ==========================================
print('방법:', ', '.join(EXPERIMENT.methods))
print(
    '타일:', EXPERIMENT.tile_size,
    '| Coarse:', EXPERIMENT.coarse_stride,
    '| Fine:', EXPERIMENT.fine_stride,
)
print('실험 조건 수:', len(EXPERIMENT.conditions))

## 6. 객체 검출 테스트 세트 자동 탐색

`data/objects/setN/target.jpg`와 `results/panorama/setN/방법/panorama.jpg`를 세트 번호로 연결한다.
파노라마 세트와 기준 물체가 추가되면 같은 방식으로 자동 확장된다.

In [ ]:
# ==========================================
# 테스트 데이터 자동 탐색
# - setN/target.jpg와 setN/panorama.jpg를 세트 번호로 연결
# ==========================================
TEST_CASES = discover_test_cases(
    target_dir=OBJECT_DIR,
    panorama_result_dir=PANORAMA_RESULT_DIR,
    methods=EXPERIMENT.methods,
)

print('자동 탐색된 테스트 세트 수:', len(TEST_CASES))
print(
    '총 실험 횟수:',
    len(TEST_CASES) * len(EXPERIMENT.methods) * len(EXPERIMENT.conditions),
)

## 7. 객체 검출 일괄 실행

각 테스트 세트와 변형 조건에서 SIFT, ORB, ALIKED를 실행한다. 기본 조건의 검출 결과와
최종 Fine Tile 매칭 이미지는 저장하고, 전체 결과는 CSV로 저장한다.

In [ ]:
# ==========================================
# 객체 검출 일괄 실행 및 CSV 저장
# ==========================================
batch_records = run_batch(
    test_cases=TEST_CASES,
    methods=EXPERIMENT.methods,
    conditions=EXPERIMENT.conditions,
    tile_size=EXPERIMENT.tile_size,
    coarse_stride=EXPERIMENT.coarse_stride,
    fine_stride=EXPERIMENT.fine_stride,
    coarse_top_k=EXPERIMENT.coarse_top_k,
    early_stop_inliers=EXPERIMENT.early_stop_inliers,
    result_root=EXPERIMENT.result_root,
    matching_root=EXPERIMENT.matching_root,
)

summary_rows = summarize_records(batch_records)
csv_paths = save_experiment_csv(
    records=batch_records,
    summary=summary_rows,
    output_dir=EXPERIMENT.metrics_dir,
)

print('실행 완료:', len(batch_records), '회')
print('원시 결과 CSV:', csv_paths['records'])
print('요약 통계 CSV:', csv_paths['summary'])

## 8. 객체 검출 정량적 결과 비교

방법·변형 조건별 성공률, 평균 탐색 시간, 평균 Inlier 비율을 비교한다.

In [ ]:
create_detection_summary_table_figure(
    summary_rows=summary_rows,
    experiment=EXPERIMENT,
    pyplot=plt,
)
plt.show()

### 8.1 방법별 객체 검출 매칭 통계

기본 조건에서 최종 선택된 Fine Tile을 기준으로 방법별 매칭 품질을 비교한다.
`평균 RANSAC 전 매칭`은 SIFT·ORB의 Ratio Test 후, ALIKED의 LightGlue 매칭 후의 수이며,
`평균 Inlier`와 `평균 Inlier 비율`은 검출 성공 케이스만 평균낸 값이다.

In [ ]:
# ==========================================
# 방법별 객체 검출 최종 Tile 매칭 통계
# - 기본 조건의 검출 성공 케이스를 기준으로 평균 매칭 품질 계산
# ==========================================
create_detection_matching_summary_figure(
    records=batch_records,
    experiment=EXPERIMENT,
    pyplot=plt,
    visual_condition='baseline',
)
plt.show()

### 8.2 조건별 성공률과 처리 시간

회전·크기·조명 변화가 검출 성공률과 평균 실행 시간에 미치는 영향을 방법별 막대그래프로 비교한다.

In [ ]:
create_detection_summary_chart_figure(
    summary_rows=summary_rows,
    experiment=EXPERIMENT,
    pyplot=plt,
)
plt.show()

## 9. 객체 검출 정성적 결과 확인

위 행은 파노라마에서 검출된 물체의 Global Polygon이고, 아래 행은 같은 방법에서 선택된
최종 Fine Tile의 Keypoint Matches이다. 수치 결과와 함께 Polygon 위치와 매칭 분포를 확인한다.

In [ ]:
# ==========================================
# 기본 조건 검출 결과 및 최종 Tile 매칭 시각화
# ==========================================
VISUAL_CONDITION = 'baseline'

create_detection_qualitative_figure(
    test_cases=TEST_CASES,
    experiment=EXPERIMENT,
    pyplot=plt,
    visual_condition=VISUAL_CONDITION,
)
plt.show()

## 10. 결론 작성 기준

1. `success_rate`로 검출 안정성과 변화 조건별 취약점을 비교한다.
2. `mean_elapsed`로 처리 속도와 실시간 적용 가능성을 해석한다.
3. 최종 Tile 매칭 통계와 Keypoint Matches 이미지로 기하학적 신뢰도를 확인한다.
4. 정확도·속도·강건성의 균형을 근거로 프로젝트 목적에 적합한 방법을 제안한다.